# LDLR variant scores → mechanistic model parameters
## Phase 1 & 2 starter notebook

This notebook covers Phases 1, 2 and the first set of Phase 3 plots from the project plan:

1. Load Tabet et al. (*Science* 2025) deep mutational scanning data: LDL uptake function scores and LDLR surface abundance scores.
2. Reproduce a few of their headline statistics as a sanity check.
3. Add structural domain annotations for each variant.
4. Compute two derived scores per variant:
   - **S** = surface abundance score (directly from Tabet's abundance map)
   - **B** = per-receptor LDL binding/uptake score (uptake residualized against abundance)
5. Make the marginal-distribution, joint-hexbin and per-position-trace plots from Phase 3.

**Before running**: download Tabet's Data S1 (LDL uptake without VLDL) and Data S2 (surface abundance) from the *Science* supplementary materials and place them in `./data/`. Adjust the filenames and column names in the loading section based on what you actually downloaded — the block makes some reasonable guesses but won't get it right without checking the files.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from statsmodels.nonparametric.smoothers_lowess import lowess

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

DATA_DIR = Path("./data")  # adjust to where you put Tabet's supplementary files
DATA_DIR.mkdir(exist_ok=True)

## 1. Load Tabet's data

Tabet et al. provide their data as supplementary files attached to the *Science* paper. The relevant files for our project are:

- **Data S1**: LDL uptake function scores (the primary function map, ~17,000 variants × ~860 positions).
- **Data S2**: LDLR cell-surface abundance scores.

Typical columns are something like `(position, wild_type_aa, variant_aa, score, sigma)` where `sigma` is the standard error. The loading helper below standardizes common column-name variants — check the printed column lists and adjust the rename map if the file uses different names.

In [ ]:
# Adjust these filenames to match what you downloaded
uptake_file = DATA_DIR / "tabet_data_S1_uptake.csv"
abundance_file = DATA_DIR / "tabet_data_S2_abundance.csv"

def load_score_file(path):
    """Load Tabet-style score file with common column-name standardization."""
    if path.suffix.lower() in (".xlsx", ".xls"):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    rename_map = {
        "pos": "position", "POS": "position", "Position": "position",
        "wt_aa": "wt", "wild_type": "wt", "WT": "wt", "ref": "wt",
        "var_aa": "alt", "variant": "alt", "ALT": "alt", "mut": "alt",
        "stderr": "sigma", "se": "sigma", "score_se": "sigma",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})
    return df

uptake = load_score_file(uptake_file)
abundance = load_score_file(abundance_file)

print(f"Uptake:    {len(uptake):>6} rows, columns: {list(uptake.columns)}")
print(f"Abundance: {len(abundance):>6} rows, columns: {list(abundance.columns)}")
uptake.head()

## 2. Merge into a single per-variant DataFrame

Each variant is uniquely identified by `(position, wt, alt)`. Merge the two score sets on this triple and keep variants present in both.

In [ ]:
variants = (
    uptake.rename(columns={"score": "uptake_score", "sigma": "uptake_se"})
    .merge(
        abundance.rename(columns={"score": "abundance_score", "sigma": "abundance_se"}),
        on=["position", "wt", "alt"],
        how="inner",
    )
)

variants["variant_type"] = np.where(
    variants["alt"] == variants["wt"], "synonymous",
    np.where(variants["alt"].isin(["*", "X", "Ter"]), "nonsense", "missense"),
)

print(f"Merged variants: {len(variants):>6}")
print(variants["variant_type"].value_counts())
variants.head()

## 3. Add structural domain annotations

LDLR domain boundaries (UniProt P01130, approximate; refine against the canonical transcript if needed). Tabet uses these regions in their Fig. 1C and Fig. 3 — cross-check against their figures if you see edge mismatches.

In [ ]:
DOMAINS = [
    ("signal",   1,   21),
    ("LA1",     22,   63),
    ("LA2",     64,  103),
    ("LA3",    104,  143),
    ("LA4",    144,  187),
    ("LA5",    188,  227),
    ("LA6",    228,  266),
    ("LA7",    267,  310),
    ("EGF-A",  311,  352),
    ("EGF-B",  353,  394),
    ("β-prop", 395,  694),  # 6-bladed propeller (B1-B6); treated as one for now
    ("EGF-C",  695,  750),
    ("linker", 751,  767),
    ("TM",     768,  789),
    ("NPxY",   790,  860),  # cytoplasmic tail incl. NPxY motif
]

def position_to_domain(pos):
    for name, start, end in DOMAINS:
        if start <= pos <= end:
            return name
    return "outside"

variants["domain"] = variants["position"].apply(position_to_domain)
variants["domain"] = pd.Categorical(
    variants["domain"],
    categories=[d[0] for d in DOMAINS] + ["outside"],
    ordered=True,
)
print(variants["domain"].value_counts().sort_index())

## 4. Sanity checks against Tabet's headline numbers

Before going further, confirm we can reproduce a few key statistics from the paper. If these don't reproduce within rounding, we have a data-processing problem to fix first.

In [ ]:
# 4a. Score distributions by variant type - should reproduce Tabet Fig. 1D
fig, ax = plt.subplots(figsize=(7, 4))
for vtype, color in [("synonymous", "C2"), ("missense", "C0"), ("nonsense", "C3")]:
    subset = variants[variants["variant_type"] == vtype]["uptake_score"].dropna()
    if len(subset) > 0:
        sns.kdeplot(subset, ax=ax, label=f"{vtype} (n={len(subset):,})",
                    color=color, fill=True, alpha=0.3)
ax.set_xlabel("LDL uptake functional score")
ax.set_ylabel("density")
ax.set_title("Score distributions by variant type (cf. Tabet Fig. 1D)")
ax.axvline(0, ls=":", c="grey", alpha=0.6)
ax.axvline(1, ls=":", c="grey", alpha=0.6)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 4b. Tabet reports r ~ 0.84 between abundance and uptake in LA1-6 (missense only)
la16 = variants[
    variants["domain"].isin(["LA1", "LA2", "LA3", "LA4", "LA5", "LA6"])
    & (variants["variant_type"] == "missense")
].dropna(subset=["uptake_score", "abundance_score"])

r = la16["uptake_score"].corr(la16["abundance_score"])
print(f"Pearson r (LA1-6 missense, n={len(la16):,}): {r:.3f}")
print(f"Tabet reports: r ~ 0.84")

fig, ax = plt.subplots(figsize=(5.5, 5))
hb = ax.hexbin(la16["abundance_score"], la16["uptake_score"],
               gridsize=40, cmap="Blues", bins="log", mincnt=1)
ax.set_xlabel("Surface abundance score")
ax.set_ylabel("LDL uptake score")
ax.set_title(f"LA1-6 missense, r = {r:.3f}")
ax.plot([0, 1.2], [0, 1.2], "k:", alpha=0.4, label="y = x")
ax.legend()
plt.colorbar(hb, ax=ax, label="log count")
plt.tight_layout()
plt.show()

## 5. Compute the derived scores S and B

- **S = abundance_score** (taken directly).
- **B = uptake residualized against abundance.** We fit a LOESS regression of uptake on abundance across all missense variants, then take the residual. Variants with B < 0 have *worse* uptake than abundance alone would predict; variants with B ≈ 0 are well-explained by abundance.

This is the simple first-pass residualization. Phase 5 replaces it with a proper joint model that doesn't rely on sequential residualization, but for the exploration phase this is fine.

In [ ]:
miss = variants[variants["variant_type"] == "missense"].dropna(
    subset=["uptake_score", "abundance_score"]
).copy()

# LOESS smoothing - frac controls smoothness; 0.3 is a reasonable default
loess_fit = lowess(
    endog=miss["uptake_score"],
    exog=miss["abundance_score"],
    frac=0.3,
    return_sorted=True,
)
loess_x, loess_y = loess_fit[:, 0], loess_fit[:, 1]
miss["uptake_predicted"] = np.interp(miss["abundance_score"], loess_x, loess_y)
miss["B"] = miss["uptake_score"] - miss["uptake_predicted"]
miss["S"] = miss["abundance_score"]

print(f"B: mean={miss['B'].mean():+.4f}, sd={miss['B'].std():.3f}, "
      f"range=[{miss['B'].min():+.2f}, {miss['B'].max():+.2f}]")
print(f"S: mean={miss['S'].mean():+.4f}, sd={miss['S'].std():.3f}, "
      f"range=[{miss['S'].min():+.2f}, {miss['S'].max():+.2f}]")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].hexbin(miss["abundance_score"], miss["uptake_score"],
               gridsize=50, cmap="Blues", bins="log", mincnt=1)
axes[0].plot(loess_x, loess_y, "r-", lw=2, label="LOESS fit")
axes[0].set_xlabel("abundance score")
axes[0].set_ylabel("uptake score")
axes[0].set_title("Raw uptake vs abundance")
axes[0].legend()

axes[1].hexbin(miss["S"], miss["B"], gridsize=50, cmap="Blues", bins="log", mincnt=1)
axes[1].axhline(0, ls=":", c="grey", alpha=0.6)
axes[1].set_xlabel("S (surface abundance)")
axes[1].set_ylabel("B (residualized uptake)")
axes[1].set_title("After residualization: B vs S")
plt.tight_layout()
plt.show()

## 6. Exploratory plots (Phase 3)

### 6a. Marginal distributions of S and B, faceted by domain (violin)

In [ ]:
# Drop signal peptide, linker and "outside"; keep biologically meaningful domains
PLOT_DOMAINS = ["LA1", "LA2", "LA3", "LA4", "LA5", "LA6", "LA7",
                "EGF-A", "EGF-B", "β-prop", "EGF-C", "TM", "NPxY"]
plot_df = miss[miss["domain"].isin(PLOT_DOMAINS)].copy()
plot_df["domain"] = pd.Categorical(plot_df["domain"], categories=PLOT_DOMAINS, ordered=True)

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
sns.violinplot(data=plot_df, x="domain", y="S", ax=axes[0],
               inner="quartile", cut=0, color="C0")
axes[0].set_ylabel("S (surface abundance)")
axes[0].axhline(1.0, ls=":", c="grey", alpha=0.6)
axes[0].axhline(0.0, ls=":", c="grey", alpha=0.6)
axes[0].set_title("S and B distributions by structural domain")

sns.violinplot(data=plot_df, x="domain", y="B", ax=axes[1],
               inner="quartile", cut=0, color="C0")
axes[1].set_ylabel("B (residualized uptake)")
axes[1].axhline(0, ls=":", c="grey", alpha=0.6)
axes[1].set_xlabel("structural domain")
plt.tight_layout()
plt.show()

**Patterns to look for**:

- **LA2 and LA6**: distributions tight near 0 on the B axis. This is our blind-spot signature - most variants in these domains look neutral for LDL uptake without VLDL, but Tabet shows that ~23% of LA2/LA6 missense variants are damaging under VLDL competition. Don't read this as "LA2/LA6 substitutions don't matter."
- **LA3, LA4, LA5, LA7**: broader, more skewed distributions on B with substantial negative tails. These are the binding-affinity-sensitive modules.
- **β-prop**: should show effect on B from the pH-sensitive residues even though most positions are tolerant.
- **NPxY**: small region, should show bimodal B distribution (some tolerated, many strongly damaging).
- **TM, EGF**: stability/abundance effects dominate - broader S distribution, narrower B distribution.

### 6b. Ridgeline alternative

If too many domains make the violin plot cramped, the ridgeline version reads more cleanly. Requires `pip install joypy`.

In [ ]:
# Uncomment if joypy is installed:
# import joypy
# fig, axes = joypy.joyplot(
#     plot_df, by="domain", column="B",
#     figsize=(9, 7), overlap=0.6, fade=False, color="C0"
# )
# plt.suptitle("B by structural domain (ridgeline)", y=1.02)
# plt.show()
print("Install joypy and uncomment the cell above for the ridgeline version.")

### 6c. Joint hexbin of (S, B) with marginal histograms

The single most informative figure for the joint structure. Look for:

- The bulk centered near (1, 0) - wild-type-like variants.
- A tail going to **low S, B ≈ 0** - variants damaged via abundance/stability alone (LA cysteines, EGF disulfides, transmembrane perturbations).
- A tail going to **S ≈ 1, low B** - variants that don't affect abundance but damage per-receptor uptake (β-propeller, NPxY motif).

If both tails are well-populated, the analytic decomposition is doing useful work. If the data collapses to a single damage axis, S and B are too correlated and we need to rethink the residualization.

In [ ]:
g = sns.jointplot(
    data=miss, x="S", y="B",
    kind="hex", gridsize=50, cmap="Blues",
    height=7, marginal_kws=dict(bins=60, fill=True),
    joint_kws=dict(bins="log", mincnt=1),
)
g.ax_joint.axhline(0, ls=":", c="grey", alpha=0.5)
g.ax_joint.axvline(1, ls=":", c="grey", alpha=0.5)
g.set_axis_labels("S (surface abundance)", "B (residualized uptake)")
g.fig.suptitle(f"Joint distribution of S and B (n = {len(miss):,} missense)", y=1.02)
plt.show()

### 6d. Per-position mean trace along the LDLR sequence

Mean S and mean B at each position, with domain bars annotated. Spikes in B without corresponding spikes in S identify your **binding-specific** positions (likely concentrated in the ApoB100-binding LA modules 3-5/7, the β-propeller and the NPxY motif). Spikes in S identify your **stability/trafficking-sensitive** positions (likely at disulfide cysteines and backbone-critical residues).

In [ ]:
pos_stats = miss.groupby("position", observed=True).agg(
    mean_S=("S", "mean"),
    mean_B=("B", "mean"),
    n=("S", "size"),
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(pos_stats["position"], pos_stats["mean_S"], color="C0", lw=0.8)
axes[0].axhline(1.0, ls=":", c="grey", alpha=0.5)
axes[0].set_ylabel("mean S")

axes[1].plot(pos_stats["position"], pos_stats["mean_B"], color="C0", lw=0.8)
axes[1].axhline(0, ls=":", c="grey", alpha=0.5)
axes[1].set_ylabel("mean B")
axes[1].set_xlabel("LDLR position")

# Light domain shading
domain_colors = sns.color_palette("husl", len(DOMAINS))
for ax in axes:
    for (name, start, end), color in zip(DOMAINS, domain_colors):
        if name in ("signal", "outside", "linker"):
            continue
        ax.axvspan(start, end, alpha=0.06, color=color)

# Domain labels on top axis
ymax = axes[0].get_ylim()[1]
for (name, start, end), color in zip(DOMAINS, domain_colors):
    if name in ("signal", "outside", "linker"):
        continue
    axes[0].text((start + end) / 2, ymax * 1.02, name,
                 ha="center", va="bottom", fontsize=8)
axes[0].set_ylim(top=ymax * 1.12)
plt.tight_layout()
plt.show()

## 7. Save the processed DataFrame for Phase 5

Phase 5's Bayesian fit consumes a clean per-variant table with S, B and any residue features we've engineered. Save it now as a Parquet file for fast reload.

In [ ]:
out_path = DATA_DIR / "variants_with_S_B.parquet"
out_cols = ["position", "wt", "alt", "domain", "variant_type",
            "uptake_score", "uptake_se", "abundance_score", "abundance_se",
            "S", "B"]
miss[out_cols].to_parquet(out_path, index=False)
print(f"Saved {len(miss):,} rows to {out_path}")

# Also save the full variants table (including synonymous/nonsense) for completeness
all_path = DATA_DIR / "variants_all.parquet"
variants.to_parquet(all_path, index=False)
print(f"Saved {len(variants):,} rows to {all_path}")

## Next steps

Plots not yet covered in this notebook (deferred to a separate one):

- **Structural overlay** on PDB 9BDE - PyMol script that colors residues by mean B and mean S. The trick is `alter <obj>, b=score_dict[resi]` followed by `spectrum b, blue_white_red`.
- **PCA of (S, B)** colored by domain and ClinVar status - useful for checking joint structure quickly.
- **ClinVar integration** - merge ClinVar P/LP and B/LB annotations onto the variant table and overlay on the joint hexbin as scatter points.

Then proceed to:

- **Phase 4** - fill in the tentative domain → parameter assignment table based on what the figures above show.
- **Phase 5** - hierarchical Bayesian fit in PyMC, consuming `variants_with_S_B.parquet`.

If any of the sanity checks in step 4 don't reproduce Tabet's headline numbers within rounding, **stop and figure out why before continuing**. The most common causes are: wrong column name mapping in the loading step, signal-peptide residues not being filtered out, or the abundance-vs-function correlation being run on the wrong variant subset.